# DelaunayInterfaces Extended Example

Interactive showcase using the Julia bindings and **GLMakie** — one figure with three tabs:

- **Tab 1** — Four tetrahedron partition variants
- **Tab 2** — Protein assemblies (4BMG, 6Z4U, 1ALY): interface + per-molecule toggles
- **Tab 3** — Brain segmentation (Mindboggle 6-label template): interface with filtration slider

All input data ships with the repo in `julia_extended_data.json`: 

Run the cells in order. The first run computes all interface surfaces (a few minutes) and
caches them in `julia_extended_cache.jld2` (gitignored); subsequent runs load from
the cache instantly. Delete the cache file to force recomputation (required after changes
to the `InterfaceSurface` struct).

---
## Setup

In [1]:
import Pkg; Pkg.activate("../julia")
using JSON
using JLD2
using GLMakie
using GeometryBasics
using LinearAlgebra

# Load the Julia bindings and visualization module
include("../julia/src/DelaunayInterfaces.jl")
using .DelaunayInterfaces

include("../julia/src/visualization.jl")

println("DelaunayInterfaces Julia bindings loaded")

DelaunayInterfaces Julia bindings loaded


---
## Data Loading & Interface Computation

Tetrahedron examples are defined in code; assembly and brain inputs come from
`julia_extended_data.json`. Computed surfaces are cached in
`julia_extended_cache.jld2`.

In [ ]:
# ── Tetrahedron geometry & colors ─────────────────────────────────────────────
const CONF_COLORS_TET = [
    RGBf(0.106, 0.620, 0.467),
    RGBf(0.851, 0.373, 0.008),
    RGBf(0.459, 0.439, 0.702),
    RGBf(0.906, 0.161, 0.541),
]

function rotate_points_tet(points, axis, angle)
    axis = normalize(axis)
    K = [0 -axis[3] axis[2]; axis[3] 0 -axis[1]; -axis[2] axis[1] 0]
    R = I + sin(angle) * K + (1 - cos(angle)) * K^2
    [Vector{Float64}(R * p) for p in points]
end
center_pts(pts) = (c = sum(pts) / length(pts); [p - c for p in pts])

UNIT_TET   = center_pts(rotate_points_tet([[0.,0.,0.],[1.,0.,0.],[0.,1.,0.],[0.,0.,1.]], [1.,0.5,0.7], π/6))
SHARED_TET = [[-1.,-1.,0.],[1.,-1.,0.],[0.,1.,-1.],[0.,1.,1.]]
SHARED_22  = center_pts(rotate_points_tet(SHARED_TET, [1.,0.,0.],  π/4))
SHARED_211 = center_pts(rotate_points_tet(SHARED_TET, [1.,0.,0.], -π/6))
TET_31     = let r = 1.0
    center_pts([[r,-1.,0.],[-r/2,-1.,r*sqrt(3)/2],[-r/2,-1.,-r*sqrt(3)/2],[0.,1.,0.]])
end

const EXAMPLES_TET = [
    (name="3-1 Partition",     points=TET_31,    colors=[1,1,1,2]),
    (name="2-2 Partition",     points=SHARED_22, colors=[1,1,2,2]),
    (name="2-1-1 Partition",   points=SHARED_211,colors=[1,1,2,3]),
    (name="1-1-1-1 Partition", points=UNIT_TET,  colors=[1,2,3,4]),
]

# ── Assembly & brain inputs from JSON ─────────────────────────────────────────
data = JSON.parsefile("julia_extended_data.json")

asm_meta = map(data["assemblies"]) do a
    (label      = a["label"],
     pts        = [Vector{Float64}(p) for p in a["points"]],
     colors     = Vector{Int}(a["colors"]),
     radii      = Vector{Float64}(a["radii"]),
     n_mol      = Int(a["n_mol"]),
     n_atoms    = Int(a["n_atoms"]),
     template_r = Vector{Float64}(a["template_radii"]))
end

brain_pts   = [Vector{Float64}(p) for p in data["brain"]["points"]]
brain_cols  = Vector{Int}(data["brain"]["colors"])
brain_radii = fill(Float64(data["brain"]["radius"]), length(brain_pts))

for m in asm_meta
    println("$(m.label): n_mol=$(m.n_mol), $(length(m.pts)) atoms")
end
println("Brain: $(length(brain_pts)) points, $(maximum(brain_cols)) labels")

# ── JLD2 cache: load if present, else compute & save ─────────────────────────
cache_path = "julia_extended_cache.jld2"

if isfile(cache_path)
    @info "Loading computed surfaces from cache…"
    cache             = load(cache_path)
    tet_surfaces      = cache["tet_surfaces"]
    asm_surfaces      = cache["asm_surfaces"]
    brain_surface     = cache["brain_surface"]
    brain_filt_levels = cache["brain_filt_levels"]
    @info "Loaded: $(length(tet_surfaces)) tet, $(length(asm_surfaces)) assembly surfaces"
else
    @info "Cache not found — computing all interfaces (first run may take a few minutes)…"

    tet_surfaces = [InterfaceSurface(ex.points, ex.colors) for ex in EXAMPLES_TET]
    println("  Tet surfaces: done")

    asm_surfaces = map(asm_meta) do m
        println("  $(m.label): computing interface surface…")
        s = InterfaceSurface(m.pts, m.colors, m.radii; weighted=true, alpha=true)
        println("    → $(length(s.vertices)) vertices")
        s
    end

    println("  Brain: computing interface ($(length(brain_pts)) points)…")
    brain_surface = InterfaceSurface(brain_pts, brain_cols, brain_radii;
                                     weighted=true, alpha=true, lower_star=true)
    brain_filt_levels = sort!(unique([val for (_, val) in brain_surface.filtration]))
    println("  Brain: $(length(brain_surface.vertices)) vertices, $(length(brain_filt_levels)) filtration levels")

    jldsave(cache_path; tet_surfaces, asm_surfaces, brain_surface, brain_filt_levels)
    @info "Cache saved → $cache_path"
end

4BMG: n_mol=2, 2268 atoms
6Z4U: n_mol=2, 1278 atoms
1ALY: n_mol=3, 3339 atoms
Brain: 900 points, 6 labels


[ Info: Loading computed surfaces from cache…
[ Info: Loaded: 4 tet, 3 assembly surfaces


---
## Tabbed Visualization

One figure with three tabs (buttons at top). Tab content is pre-built; all heavy
computation was done above.

In [5]:
# ── Pre-compute brain filtration meshes (fast once surface is cached) ─────────
println("Pre-computing $(length(brain_filt_levels)) brain filtration meshes…")
_bm              = [generate_colored_mesh(brain_surface; max_value=lvl) for lvl in brain_filt_levels]
filt_mesh_list   = first.(_bm)
filt_color_list  = last.(_bm)
_fc              = last(filt_color_list)
colorrange_brain = isempty(_fc) ? (0.0, 1.0) : (minimum(_fc), maximum(_fc))
n_colors_brain   = maximum(brain_cols)
println("  Done.")

# ── Pre-compute free simplex data at each filtration level ────────────────────
# lower_star=true guarantees face ≤ coface, so free edges and isolated vertices
# are meaningful at intermediate levels — triangles drop first, then edges, then vertices.
function _free_at(surface, max_value)
    barycenters    = [Point3f(v...) for v in surface.vertices]
    vertex_vals    = Dict{Int,Float64}()
    edges          = Set{Tuple{Int32,Int32}}()
    triangle_edges = Set{Tuple{Int32,Int32}}()
    edge_verts     = Set{Int32}()
    for (simplex, val) in surface.filtration
        val > max_value && continue
        if length(simplex) == 1
            vertex_vals[Int(simplex[1])] = val
        elseif length(simplex) == 2
            push!(edges, minmax(Int32(simplex[1]), Int32(simplex[2])))
        elseif length(simplex) == 3
            for i in 1:3, j in (i+1):3
                push!(triangle_edges, minmax(Int32(simplex[i]), Int32(simplex[j])))
            end
        end
    end
    free_edges = setdiff(edges, triangle_edges)
    fe_pts = Point3f[]; fe_cols = Float64[]
    for (i, j) in free_edges
        push!(fe_pts, barycenters[Int(i)], barycenters[Int(j)])
        push!(fe_cols, get(vertex_vals, Int(i), 0.0), get(vertex_vals, Int(j), 0.0))
        push!(edge_verts, i, j)
    end
    for (i, j) in triangle_edges; push!(edge_verts, i, j); end
    fv_pts = Point3f[]; fv_cols = Float64[]
    for (simplex, val) in surface.filtration
        val > max_value && continue
        if length(simplex) == 1 && !(Int32(simplex[1]) in edge_verts)
            push!(fv_pts, barycenters[Int(simplex[1])])
            push!(fv_cols, val)
        end
    end
    fe_pts, fe_cols, fv_pts, fv_cols
end

println("Pre-computing brain free simplex data…")
_bfs = [_free_at(brain_surface, lvl) for lvl in brain_filt_levels]
filt_free_edge_pts  = [x[1] for x in _bfs]
filt_free_edge_cols = [x[2] for x in _bfs]
filt_free_vert_pts  = [x[3] for x in _bfs]
filt_free_vert_cols = [x[4] for x in _bfs]
println("  Done.")

# ── Visibility helper: recursively hide/show all blocks in a GridLayout ───────
function set_gl_visible!(gl::GridLayout, vis::Bool)
    for item in contents(gl)
        if item isa GridLayout
            set_gl_visible!(item, vis)
        elseif item isa LScene
            item.scene.visible[]      = vis
            item.blockscene.visible[] = vis
        else
            try; item.blockscene.visible[] = vis; catch; end
        end
    end
end

# ── Helper: draw one tetrahedron panel ────────────────────────────────────────
function draw_tet_panel!(scene, points, colors, surf)
    n = length(points)
    edge_pts = Point3f[]; edge_cols = RGBAf[]
    for i in 1:n, j in (i+1):n
        c1 = CONF_COLORS_TET[mod1(colors[i], 4)]
        c2 = CONF_COLORS_TET[mod1(colors[j], 4)]
        push!(edge_pts, Point3f(points[i]...), Point3f(points[j]...))
        push!(edge_cols, RGBAf(c1), RGBAf(c2))
    end
    linesegments!(scene, edge_pts; color=edge_cols, linewidth=2)

    verts     = surf.vertices
    triangles = [Int.(s) for (s, _) in surf.filtration if length(s) == 3]
    if !isempty(triangles) && !isempty(verts)
        pts_gb = [Point3f(v...) for v in verts]
        mesh!(scene, GeometryBasics.Mesh(pts_gb, [TriangleFace(t...) for t in triangles]);
              color=RGBf(0.7,0.7,0.7), shading=NoShading, depth_shift=5f-4)
        tri_edge_pts = Point3f[]
        for t in triangles
            p1, p2, p3 = pts_gb[t[1]], pts_gb[t[2]], pts_gb[t[3]]
            push!(tri_edge_pts, p1,p2, p2,p3, p3,p1)
        end
        linesegments!(scene, tri_edge_pts; color=:black, linewidth=1.5)
    end
    meshscatter!(scene, [Point3f(p...) for p in points];
        color=[CONF_COLORS_TET[mod1(c, 4)] for c in colors], markersize=0.05, shading=NoShading)
end

# ── Tab 1: four tetrahedron partition variants ────────────────────────────────
function draw_tab1!(layout)
    for (ex, surf, pos) in zip(EXAMPLES_TET, tet_surfaces, [(1,1),(1,2),(2,1),(2,2)])
        sc = LScene(layout[pos...]; show_axis=false)
        draw_tet_panel!(sc, ex.points, ex.colors, surf)
        Label(layout[pos[1], pos[2], Top()], ex.name; fontsize=16, padding=(0,0,5,0))
    end
end

# ── Tab 2: protein assemblies (4BMG, 6Z4U, 1ALY) side by side ────────────────
function draw_tab2!(layout)
    conf_grad = cgrad(:Dark2_3, 3, categorical=true)
    for (col, (meta, surf)) in enumerate(zip(asm_meta, asm_surfaces))
        n_mol = meta.n_mol
        n_atm = meta.n_atoms

        Label(layout[1, col], meta.label; fontsize=20, font=:bold, tellwidth=false)
        sc = LScene(layout[2, col]; show_axis=false)
        draw_interface!(sc, surf; show_wireframe=false)
        draw_free_simplices!(sc, surf)

        mol_plots = Vector{Any}(undef, n_mol)
        for j in 1:n_mol
            idx = ((j-1)*n_atm+1):(j*n_atm)
            c   = conf_grad[mod1(j, 3)]
            mol_plots[j] = meshscatter!(sc,
                [Point3f(meta.pts[i]...) for i in idx];
                markersize = Float32.(meta.template_r .+ 1.4) .* 0.5f0,
                color      = RGBAf(c.r, c.g, c.b, 0.35f0),
                shading    = NoShading)
        end

        tg_row = GridLayout(layout[3, col]; tellwidth=false)
        for j in 1:n_mol
            tg = Toggle(tg_row[1, j]; active=true)
            Label(tg_row[2, j], "Mol $j"; fontsize=13)
            let plot = mol_plots[j]
                on(tg.active) do val; plot.visible[] = val; end
            end
        end
    end
end

# ── Tab 3: brain segmentation with filtration & point-size sliders ────────────
function draw_tab3!(layout)
    cmap_b = cgrad(:tab10, n_colors_brain, categorical=true)
    Label(layout[1, 1], "Brain Segmentation — 6 Regions (Alpha Complex, lower star)";
          fontsize=20, tellwidth=false)
    sc = LScene(layout[2, 1]; show_axis=false)

    sl_filt = Slider(layout[3, 1];
        range=1:length(brain_filt_levels), startvalue=length(brain_filt_levels))
    Label(layout[4, 1],
        @lift("Filtration level: $(round(brain_filt_levels[$(sl_filt.value)], digits=3))");
        fontsize=13, tellwidth=false)

    # Triangles
    mesh!(sc, @lift(filt_mesh_list[$(sl_filt.value)]);
        color=@lift(filt_color_list[$(sl_filt.value)]),
        colorrange=colorrange_brain, colormap=:viridis, shading=NoShading)

    # Free edges: in the alpha complex but not yet bounded by a triangle
    linesegments!(sc, @lift(filt_free_edge_pts[$(sl_filt.value)]);
        color      = @lift(filt_free_edge_cols[$(sl_filt.value)]),
        colormap   = :viridis,
        colorrange = colorrange_brain,
        linewidth  = 2)

    # Isolated vertices: in the alpha complex but not yet connected by any edge
    scatter!(sc, @lift(filt_free_vert_pts[$(sl_filt.value)]);
        color      = @lift(filt_free_vert_cols[$(sl_filt.value)]),
        colormap   = :viridis,
        colorrange = colorrange_brain,
        markersize = 8)

    sg = GridLayout(layout[5, 1]; tellwidth=false)
    Label(sg[1, 1], "Point size:"; fontsize=13, tellwidth=false)
    sl_size = Slider(sg[1, 2]; range=0.1f0:0.5f0:30.0f0, startvalue=1.0f0)

    for j in 1:n_colors_brain
        idx = findall(==(j), brain_cols)
        c   = cmap_b[j]
        meshscatter!(sc, [Point3f(brain_pts[i]...) for i in idx];
            markersize = @lift($(sl_size.value) * 3.0f0 * 0.35f0),
            color      = RGBAf(c.r, c.g, c.b, 0.4f0),
            shading    = NoShading)
    end
end

# ── Main tabbed figure ────────────────────────────────────────────────────────
fig = Figure(size=(1400, 950), backgroundcolor=:white)

tab_labels = ["Tetrahedra Partitions", "Protein Assemblies", "Brain Segmentation"]
active_tab = Observable(1)

# Tab buttons (row 1)
btn_gl = fig[1, 1] = GridLayout(tellwidth=false)
btns   = [Button(btn_gl[1, i]; label=tab_labels[i], fontsize=15) for i in 1:3]

# Pre-build all three tabs in rows 2, 3, 4
tab_gls = [fig[i+1, 1] = GridLayout() for i in 1:3]
draw_tab1!(tab_gls[1])
draw_tab2!(tab_gls[2])
draw_tab3!(tab_gls[3])

# Collapse and hide tabs 2 and 3 initially
for j in 2:3
    rowsize!(fig.layout, j+1, Fixed(0))
    set_gl_visible!(tab_gls[j], false)
end

# Tab switching: hide+collapse old, show+expand new
for (i, btn) in enumerate(btns)
    on(btn.clicks) do _
        active_tab[] == i && return
        prev = active_tab[]
        active_tab[] = i
        rowsize!(fig.layout, prev+1, Fixed(0))
        set_gl_visible!(tab_gls[prev], false)
        rowsize!(fig.layout, i+1, Auto())
        set_gl_visible!(tab_gls[i], true)
    end
end

display(fig)

Pre-computing 6456 brain filtration meshes…
  Done.
Pre-computing brain free simplex data…
  Done.


GLMakie.Screen(...)